# Ensemble Learning

## Setup stage

Mounting Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Cloning git repository to access the codebase.

In [ ]:
!git clone -b feat/data-augm --single-branch https://github.com/deadPixelsGreta/xAI-proj-m-ws2526.git

Copying dataset from the Google Drive to the current local disk of VM.

In [ ]:
import shutil
import os
from tqdm import tqdm

source_path = "/content/drive/MyDrive/ImageNetSubset/"
destination_path = "/content/xAI-proj-m-ws2526/datasets/"

# If the destination directory exists, remove it first
if os.path.exists(destination_path):
    print(f"Removing existing directory: {destination_path}")
    shutil.rmtree(destination_path)

# Custom copy function with tqdm
def copytree_with_tqdm(src, dst):
    # Calculate total number of items (files and directories) to copy for tqdm
    total_items = 0
    for dirpath, dirnames, filenames in os.walk(src):
        total_items += len(dirnames) # for directories
        total_items += len(filenames) # for files

    # Ensure the destination root directory exists
    os.makedirs(dst, exist_ok=True)

    with tqdm(total=total_items, unit="item", desc=f"Copying {os.path.basename(src)}") as pbar:
        for dirpath, dirnames, filenames in os.walk(src):
            # Create subdirectories in destination
            relative_path = os.path.relpath(dirpath, src)
            current_dst_dir = os.path.join(dst, relative_path)

            for dirname in dirnames:
                dest_dir = os.path.join(current_dst_dir, dirname)
                os.makedirs(dest_dir, exist_ok=True)
                pbar.update(1)

            # Copy files
            for filename in filenames:
                src_file = os.path.join(dirpath, filename)
                dst_file = os.path.join(current_dst_dir, filename)
                shutil.copy2(src_file, dst_file)
                pbar.update(1)

# Call the custom copy function
copytree_with_tqdm(source_path, destination_path)

Setup the root directory for the project. IMPORTANT for module imports.

In [2]:
import sys, os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    """Walk upward to find the outermost folder containing common project markers."""
    markers = {".git", "requirements.txt", "setup.py", "pyproject.toml"}
    root = None
    for parent in [start, *start.parents]:
        if any((parent / m).exists() for m in markers):
            root = parent  # keep going to prefer the outermost match
    return root or start

# Dynamically get the name of the cloned repository if it exists
cloned_repo_name = "xAI-proj-m-ws2526"
cloned_repo_path = Path.cwd() / cloned_repo_name

# If the cloned repository exists as a subdirectory, change into it
if cloned_repo_path.is_dir():
    os.chdir(cloned_repo_path)

# Now, find the project root from within the repository (or its parent if already there)
ROOT = find_project_root(Path.cwd()).resolve()

# Ensure we are in the identified project root
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
print("cwd:", Path.cwd())
print("root on sys.path:", str(ROOT) in sys.path)

cwd: C:\Users\ba081274\Documents\xAI\xAI-proj-m-ws2526
root on sys.path: True


In [ ]:
# Initialize conda for PowerShell
# Do this in the TERMINAL
& "C:\ProgramData\anaconda3\Scripts\conda.exe" init powershell

In [ ]:
!conda create -n xai-proj python=3.11 -y

In [5]:
# Install dependencies
!pip install -r experiments/requirements.txt --quiet

In [7]:
import wandb

# Login to WandB - this will prompt you to enter your API key
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\ba081274\_netrc
wandb: Currently logged in as: stephpark1plus1 (stephpark1plus1-uni) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
Use this path on Windows: C:\Users\ba081274\Downloads\ImageNetSubset\ImageNetSubset\

In [6]:
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet18 --seed 0

RESNET18 Training on ImageNetSubset

Random seed: 0
Device: CUDA (NVIDIA RTX A4000)

 Dataset Summary:
   Training samples: 13000
   Validation samples: 500
   Classes: ['binder', 'coffee_mug', 'computer_keyboard', 'mouse', 'notebook', 'remote_control', 'soup_bowl', 'teapot', 'toilet_tissue', 'wooden_spoon']
   Number of classes: 10

Loading pretrained resnet18 weights...
resnet18 ready with 10 output classes

 Training Configuration:
   Epochs: 5
   Batch size: 16
   Learning rate: 0.001
   Momentum: 0.9
   Weight decay: 0.0001
   Pretrained: True
   Save directory: experiments/checkpoints
   Wandb logging: True


Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\ba081274\Documents\xAI\xAI-proj-m-ws2526\experiments\scripts\train.py", line 264, in <module>
    main()
  File "C:\Users\ba081274\Documents\xAI\xAI-proj-m-ws2526\experiments\scripts\train.py", line 248, in main
    results = train(
              ^^^^^^
  File "C:\Users\ba081274\Documents\xAI\xAI-proj-m-ws2526\experiments\src\training\trainer.py", line 138, in train
    wandb.init(
  File "c:\Users\ba081274\.conda\envs\xai\Lib\site-packages\wandb\sdk\wandb_init.py", line 1594, in init
    get_sentry().reraise(e)
  File "c:\Users\ba081274\.conda\envs\xai\Lib\site-packages\wandb\analytics\sentry.py", line 190, in reraise
    raise exc.with_traceback(tb)
  File "c:\Users\ba081274\.conda\envs\xai\Lib\site-packages\wandb\sdk\wandb_init.py", line 1515, in init
    wi.maybe_login(init_settings)
  File "c:\Users\ba081274\.conda\envs\xai\Li

---

## Training Stage

In [ ]:
# Train ResNet-18 (via config)
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet18 --seed 0
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet18 --seed 1
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet18 --seed 2


In [ ]:
# Train ResNet-34 (via config)
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet34 --seed 0
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet34 --seed 1
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model resnet34 --seed 2

In [ ]:
# Train EfficientNet-B0 (via config)
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model efficientnet_b0 --seed 0
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model efficientnet_b0 --seed 1
!python -m experiments.scripts.train --config experiments/configs/default.yaml --model efficientnet_b0 --seed 2


---

## Validation Stage

In [ ]:
# Run ensemble evaluation on a dataset
!python -m experiments.scripts.inference --evaluate --data-dir datasets --wandb

In [ ]:
# Upload a test image or use sample
!python -m experiments.scripts.inference --image datasets/test_image.jpg --show-individual

---

## Hyperparameter Tuning with WandB Sweeps

This section demonstrates professional hyperparameter tuning using WandB's built-in Sweep feature. We'll optimize the learning rate using Bayesian optimization to find the best configuration efficiently.

### What is a WandB Sweep?
- **Automated hyperparameter optimization** using various search strategies (grid, random, Bayesian)
- **Early stopping** to save compute resources
- **Parallel runs** can be executed across multiple machines
- **Interactive dashboard** to visualize and compare results


In [ ]:
import subprocess
import json

# Initialize WandB sweep for learning rate optimization
# This creates a sweep configuration and initializes it
sweep_config_path = "experiments/configs/sweep_learning_rate.yaml"

print("=" * 70)
print("INITIALIZING WANDB SWEEP FOR LEARNING RATE OPTIMIZATION")
print("=" * 70)
print("\nSweep Configuration:")
print("  - Method: Bayesian Optimization (efficient hyperparameter search)")
print("  - Metric: Validation Accuracy (to maximize)")
print("  - Learning Rate Range: 1e-5 to 0.1 (log scale)")
print("  - Batch Sizes: [16, 32, 64]")
print("  - Early Stopping: Hyperband (stops unpromising runs early)")
print("\nNote: You can initialize the sweep with:")
print(f"  wandb sweep {sweep_config_path}")
print("=" * 70)


In [ ]:
# Step 1: Initialize the sweep (one-time setup)
# Uncomment the line below and run to create a new sweep
# This will return a SWEEP_ID that you'll use for running agents

import os
os.chdir(ROOT)  # Ensure we're in the project root

# Initialize sweep - only run this once!
sweep_init_cmd = ["wandb", "sweep", "experiments/configs/sweep_learning_rate.yaml"]
print("\nInitializing sweep...")
print(f"Command: {' '.join(sweep_init_cmd)}")
print("\nUncomment and run the next cell to initialize the sweep with WandB.")


In [ ]:
# STEP 1: Uncomment and run this cell to initialize the sweep
# result = subprocess.run(sweep_init_cmd, capture_output=True, text=True)
# print(result.stdout)
# if result.returncode != 0:
#     print("Error:", result.stderr)

# After running, you'll get a SWEEP_ID that looks like: "project-name/sweep-id-hash"
# Save this ID for the next step


### Option 1: Run Sweep via Terminal (Recommended for longer sweeps)

Open a terminal and run:
```bash
# Initialize the sweep (one-time)
wandb sweep experiments/configs/sweep_learning_rate.yaml

# Then run an agent (replace SWEEP_ID with the ID from above)
wandb agent your-entity/imagenet-subset-randSeed/SWEEP_ID
```

You can run multiple agents in parallel on different terminals to speed up the sweep!


### Option 2: Programmatic Sweep with Custom Training Loop

For more control, you can run a custom training loop that integrates with WandB:


In [ ]:
def train_with_wandb_sweep(config):
    """
    Wrapper function for sweep integration.
    WandB will call this function with different parameter configurations.
    """
    import os
    import sys
    from pathlib import Path
    
    # Set up path
    os.chdir(ROOT)
    
    # Import after path setup
    from experiments.src.utils import get_device, set_seed
    from experiments.src.utils.device import get_device_name
    from experiments.src.models import create_model
    from experiments.src.data import create_data_loaders
    from experiments.src.training import train
    
    device = get_device()
    
    # Extract parameters from WandB config (with defaults from sweep config)
    lr = config.get("lr", 0.001)
    batch_size = config.get("batch_size", 32)
    epochs = config.get("epochs", 10)
    seed = config.get("seed", 0)
    model_name = config.get("model", "resnet18")
    data_dir = config.get("data_dir", "C:\\Users\\ba081274\\Downloads\\ImageNetSubset\\ImageNetSubset")
    
    # Set seed for reproducibility
    set_seed(seed)
    
    # Create data loaders with sweep batch size
    train_loader, val_loader, num_classes = create_data_loaders(
        data_dir, batch_size, num_workers=0
    )
    
    # Create model
    model = create_model(model_name, num_classes, pretrained=True, device=device)
    
    # Training config
    train_config = {
        "epochs": epochs,
        "lr": lr,
        "momentum": 0.9,
        "weight_decay": 1e-4,
        "batch_size": batch_size,
        "num_classes": num_classes,
    }
    
    # WandB config
    wandb_config = {
        "project": "imagenet-subset-randSeed",
        "run_name": None,  # WandB auto-generates
        "entity": None,
        "tags": ["sweep", f"lr-tuning"],
        "notes": f"LR: {lr}, Batch: {batch_size}",
        "mode": "online",
    }
    
    # Train with WandB logging
    results = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        config=train_config,
        device=device,
        save_dir="experiments/checkpoints",
        model_name=f"{model_name}_sweep",
        wandb_enabled=True,
        wandb_config=wandb_config,
    )
    
    return results

print("✓ Sweep training function defined successfully!")
print("\nYou can now run this function with WandB's agent system.")


### Monitoring & Analysis
After starting your sweep, visit your WandB project to:
1. **View all runs**: See metrics for each configuration tried
2. **Compare parameters**: Identify which learning rates performed best
3. **Parallel coordinates plot**: Visualize relationships between parameters and performance
4. **Best run**: WandB automatically tracks the best performing configuration
5. **Export results**: Download hyperparameter data for further analysis


## Script for saving best models in Google Drive

In [ ]:
# Define source and destination paths
# We check experiments/checkpoints (relative to project root) first
source_dir = Path("experiments/checkpoints")

# Check for fallback paths if the default doesn't exist (e.g., if strictly using /chechpoints)
if not source_dir.exists():
    if Path("/chechpoints").exists(): # Handling the specific path mentioned
        source_dir = Path("/chechpoints")
    elif Path("/checkpoints").exists(): # Handling potential typo correction
        source_dir = Path("/checkpoints")

# Destination folder on the mounted drive
# You can change "saved_checkpoints" to your preferred folder name
dest_dir = Path("/content/drive/MyDrive/saved_checkpoints")

print(f"Source Directory: {source_dir}")
print(f"Destination Directory: {dest_dir}")

# Create destination directory if it doesn't exist
os.makedirs(dest_dir, exist_ok=True)

# Copy .pth files
if source_dir.exists():
    pth_files = list(source_dir.glob("*.pth"))
    
    if not pth_files:
        print("No .pth files found in source directory.")
    else:
        print(f"Found {len(pth_files)} .pth files to copy.")
        
        for file_path in pth_files:
            try:
                shutil.copy2(file_path, dest_dir / file_path.name)
                print(f"Successfully copied: {file_path.name}")
            except Exception as e:
                print(f"Error copying {file_path.name}: {e}")
else:
    print(f"Source directory {source_dir} not found. Please check the path.")